## Analysis of EuRepoC Reported Cyber Incidents 
# Goal is to use NLP and machine learning to predict target countries of cyber attacks
Dataset: eurepoc_dyadic_dataset_0_1.csv

Which latent types of cyber operations emerge from incident descriptions, and how do these topics differ in intensity, impact, and political response?

1. Clean attack descriptions
2. Generate sentence embeddings (Sentence-BERT)
3. Topic modeling (BERTopic)
4. Topic discovery
5. Relationship with weighted intensity


Old Idea 1: Map the evolution of state-sponsord cyber operations from 2000-2024 using BERTopic and transformer embeddings.

1. Clean attack descriptions
2. generate sentence embeddings (Sentence-BERT)
3. Topic modeling (BERTopic)
4. Extract countries and organizations (spaCy)
5. Track topics over time
6. Build attack-type classifier
7. Visualize geopolitical networks

Old Idea 2: Predict receiver country based on initiating country, operation type, source disclosure (reported by whom), data theft, disruption, hijacking, ransomware, doxxing, physical effects (spatial, temporal), weighted cyber intensity (higher weightings for politically motivated attacks), impact indicator (sum of individual impact scores), political impact, intelligence impact, economic impact, number political responses, number legal responses

In [144]:
import pandas as pd
import html
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

In [106]:
# Read in data 
url = "https://zenodo.org/records/14965395/files/eurepoc_dyadic_dataset_0_1.csv?download=1"

dyadic_data_full = pd.read_csv(url)

print(dyadic_data_full.shape)
print(dyadic_data_full.head())

(4296, 57)
   dyad_id initiator_country initiator_alpha_2 receiver_country  \
0        0       Afghanistan                AF      Afghanistan   
1        1       Afghanistan                AF         Pakistan   
2        1       Afghanistan                AF         Pakistan   
3        2           Algeria                DZ          Algeria   
4        2           Algeria                DZ          Algeria   

  receiver_country_alpha_2_code  incident_id  \
0                            AF         4278   
1                            PK          488   
2                            PK          499   
3                            DZ          525   
4                            DZ         1183   

                                                name  \
0  TalibLeaks Hacked and Leaked Documents from Ta...   
1               Afghan Cyber Army attack on Pakistan   
2       Afghan Cyber Army attack on Pakistan Part II   
3            Over-X vs. Algerian ministry of housing   
4               N

In [107]:
vars_to_check = [
    "impact_indicator_score",
    "weighted_intensity",
    "unweighted_intensity",
    "economic_impact",
    "functional_impact",
    "intelligence_impact"
]

for var in vars_to_check:
    print(f"\n{'='*60}")
    print(f"Value counts for: {var}")
    print(f"{'='*60}")
    print(dyadic_data_full[var].value_counts(dropna=False).sort_index())


Value counts for: impact_indicator_score
impact_indicator_score
0     2002
1       10
2       11
3       25
4       47
5      384
6      609
7      467
8      255
9      279
10     102
11      59
12      18
13      20
14       6
15       2
Name: count, dtype: int64

Value counts for: weighted_intensity
weighted_intensity
0      125
1     1284
2      594
3     1294
4      751
5      146
6       91
7        1
8        4
9        5
11       1
Name: count, dtype: int64

Value counts for: unweighted_intensity
unweighted_intensity
0     103
1    1296
2     600
3    1288
4     793
5     147
6      68
7       1
Name: count, dtype: int64

Value counts for: economic_impact
economic_impact
=< 10 Mio               56
> 10 Mio - 100 Mio      31
> 100 Mio - 1 bn         5
Not available         4204
Name: count, dtype: int64

Value counts for: functional_impact
functional_impact
Day (< 24h)                           273
Days (< 7 days)                       405
Months                                

In [108]:
# Isolate columns of interest
drop_vars = ["added_to_db","updated_at","attribution_id",\
    "initiator_alpha_2","receiver_country_alpha_2_code",
    "Data theft","Data theft & Doxing","Disruption",
    "Hijacking with Misuse","Hijacking without Misuse",
    "Ransomware","Not available"]

attack_labels = ["Data theft","Data theft & Doxing", \
    "Disruption","Hijacking with Misuse","Hijacking without Misuse",
    "Ransomware","Not available"]
attack_data = dyadic_data_full[["incident_id"] + attack_labels].copy()

dyadic_data = dyadic_data_full.drop(columns=drop_vars,\
    errors="ignore")

print(f"Original shape: {dyadic_data_full.shape}")
print(f"New shape: {dyadic_data.shape}")

print("\nRemaining columns:")
print(dyadic_data.columns.tolist())


Original shape: (4296, 57)
New shape: (4296, 45)

Remaining columns:
['dyad_id', 'initiator_country', 'receiver_country', 'incident_id', 'name', 'description', 'start_date', 'end_date', 'source_disclosure', 'operation_type', 'impact_indicator_score', 'impact_indicator_label', 'unweighted_intensity', 'weighted_intensity', 'number_attributions', 'number_political_responses', 'number_legal_responses', 'casualties', 'initiator_name', 'initiator_category', 'initiator_subcategory', 'receiver_id', 'receiver_name', 'receiver_category', 'receiver_subcategory', 'receiver_regions', 'offline_conflict_issue', 'offline_conflict_name', 'offline_conflict_intensity', 'offline_conflict_intensity_subcode', 'cyber_conflict_issue', 'physical_effects_spatial', 'physical_effects_temporal', 'target_multiplier', 'functional_impact', 'intelligence_impact', 'economic_impact', 'economic_impact_value', 'economic_impact_currency', 'affected_entities', 'affected_entities_value', 'affected_eu_countries', 'affected_eu

In [109]:
# Check for missing values in descriptions
print(dyadic_data["description"].isnull().sum())
print(dyadic_data[dyadic_data["description"]==""])

0
Empty DataFrame
Columns: [dyad_id, initiator_country, receiver_country, incident_id, name, description, start_date, end_date, source_disclosure, operation_type, impact_indicator_score, impact_indicator_label, unweighted_intensity, weighted_intensity, number_attributions, number_political_responses, number_legal_responses, casualties, initiator_name, initiator_category, initiator_subcategory, receiver_id, receiver_name, receiver_category, receiver_subcategory, receiver_regions, offline_conflict_issue, offline_conflict_name, offline_conflict_intensity, offline_conflict_intensity_subcode, cyber_conflict_issue, physical_effects_spatial, physical_effects_temporal, target_multiplier, functional_impact, intelligence_impact, economic_impact, economic_impact_value, economic_impact_currency, affected_entities, affected_entities_value, affected_eu_countries, affected_eu_countries_value, affected_third_countries, affected_third_countries_value]
Index: []

[0 rows x 45 columns]


In [128]:
# Incident-level dataset for NLP
# Descriptions can be duplicated because of the dyadic nature of the dataset. 
# One incident may span several rows if it affected several countries. 
incident_text_data = (
    dyadic_data
    .drop_duplicates(subset=["incident_id"])
    .copy()
)
print(incident_text_data.head())


   dyad_id initiator_country receiver_country  incident_id  \
0        0       Afghanistan      Afghanistan         4278   
1        1       Afghanistan         Pakistan          488   
2        1       Afghanistan         Pakistan          499   
3        2           Algeria          Algeria          525   
4        2           Algeria          Algeria         1183   

                                                name  \
0  TalibLeaks Hacked and Leaked Documents from Ta...   
1               Afghan Cyber Army attack on Pakistan   
2       Afghan Cyber Army attack on Pakistan Part II   
3            Over-X vs. Algerian ministry of housing   
4               North African Fox Espionage campaign   

                                         description           start_date  \
0  On 7 February 2025, a group of hackers, callin...  2024-01-01 00:00:00   
1  Afghan hackers deface six Pakistani government...  2013-07-11 00:00:00   
2  Afghan hackers hack the webpage of the Pakista...  2013-

In [129]:
# Clean description
incident_text_data["description"] = (
    incident_text_data["description"] 
    .fillna("")
    .astype(str)
    .apply(html.unescape)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [130]:
# Missing descriptions
print(incident_text_data["description"].isna().sum())

# Empty descriptions
print((incident_text_data["description"] == "").sum())

# Length distribution
incident_text_data["n_words"] = (
    incident_text_data["description"]
    .str.split()
    .str.len()
)

print(incident_text_data["n_words"].describe())

incident_text_data.loc[
    incident_text_data["n_words"] < 5,
    ["incident_id", "description"]
].head(20)


0
0
count    2957.000000
mean       80.144065
std        69.850175
min         3.000000
25%        29.000000
50%        65.000000
75%       112.000000
max       975.000000
Name: n_words, dtype: float64


,incident_id,description
1366,361,Taliban website hacked
2433,370,Israeli Government Site Hacked
4058,358,The Unknowns' hack NASA


In [131]:
# Check how many short descriptions I'd lose if I filtered them out
print((incident_text_data["n_words"] < 5).sum())
print((incident_text_data["n_words"] < 10).sum())

3
105


In [132]:
# Filter out descriptions with less than five words
incident_text_data = incident_text_data[incident_text_data["n_words"] > 5]

In [133]:
# Sentence embeddings with BERT
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = model.encode(
    incident_text_data["description"].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 92/92 [00:02<00:00, 40.20it/s]


In [134]:
print(embeddings.shape)

(2942, 384)


In [147]:
# Get rid of stopwords, BERT wasn't using them effectively
vectorizer_model = CountVectorizer(
    stop_words="english",
    min_df=5,
    ngram_range=(1, 2)
)

# BERTopic analysis 
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    min_topic_size=20,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(
    incident_text_data["description"],
    embeddings
)

2026-08-12 15:45:44,958 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-12 15:45:48,160 - BERTopic - Dimensionality - Completed ✓
2026-08-12 15:45:48,161 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-12 15:45:48,221 - BERTopic - Cluster - Completed ✓
2026-08-12 15:45:48,222 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-12 15:45:48,324 - BERTopic - Representation - Completed ✓


In [148]:
# Examine topics
topic_info[["Topic", "Count", "Representation"]]\
    .style.set_table_attributes(
        'style="display:block;max-height:500px;overflow:auto;"'
    )

,Topic,Count,Representation
0,-1,856,"['the', 'and', 'to', 'of', 'in', 'on', 'by', 'group', 'that', 'as']"
1,0,964,"['the', 'of', 'and', 'to', 'on', 'data', 'in', 'ransomware', 'information', 'that']"
2,1,220,"['ddos', 'the', 'of', 'attacks', 'attack', 'websites', 'on', 'website', 'to', 'and']"
3,2,145,"['russian', 'the', 'of', 'and', 'to', 'ukrainian', 'on', 'that', 'in', 'data']"
4,3,106,"['iranian', 'the', 'of', 'in', 'to', 'and', 'iran', 'spyware', 'on', 'israeli']"
5,4,91,"['in', 'the', 'and', 'to', 'chinese', 'group', 'malware', 'as', 'with', 'apt']"
6,5,88,"['pakistani', 'indian', 'websites', 'website', 'hacked', 'defaced', 'pakistan', 'of', 'government', 'hackers']"
7,6,74,"['the', 'cryptocurrency', 'million', 'to', 'funds', 'of', 'on', 'crypto', 'exchange', 'in']"
8,7,60,"['chinese', 'government', 'of', 'china', 'the', 'and', 'to', 'hackers', 'in', 'into']"
9,8,57,"['korean', 'north', 'south', 'the', 'korea', 'and', 'in', 'to', 'of', 'kimsuky']"
